In [1]:
%load_ext autoreload
%autoreload 2

import itertools
from math import prod

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm

from assignment_1.modules.models import (
    bpr_scorer,
    cand_features,
    fit_bpr,
    fit_pair_context,
    pair_features,
)
from assignment_1.modules.util import (
    calc_pair_accuracy,
    clip_timestamps,
    load_csv,
    make_pair_validation,
    time_bin_edges,
    time_bins,
)

DATA_DIR = "../data"  # notebook runs from notebooks/

In [2]:
TASK_TRAIN_FILE = "tasks_3_4_train.csv"
TASK_TEST_FILE = "task_4_test.csv"
train = load_csv(TASK_TRAIN_FILE, DATA_DIR)
test = load_csv(TASK_TEST_FILE, DATA_DIR)

In [3]:
train.head(5)

,user,item,timestamp,rating
0,u1838647363,i0846907466,1360914337,3
1,u7093796076,i2832259763,1472021070,4
2,u4779464116,i9244553928,1480231213,5
3,u2107634551,i2346931943,1343026318,4
4,u0480528309,i3693294311,1405148743,4


In [4]:
test.head(5)

,user,timestamp,item1,item2,which
0,u9337049416,1455001830,i0277476004,i3096330769,NaN
1,u0128064500,1405148935,i1691394674,i7145553690,NaN
2,u2287415013,1488186442,i7337593393,i6009132051,NaN
3,u2406803247,1346135928,i3019728190,i8275841250,NaN
4,u1146240669,1312185505,i8121318922,i9624491588,NaN


In [5]:
first_seen = train.groupby("item")["timestamp"].min()     # earliest interaction per book
t = test["timestamp"]
early1 = test["item1"].map(first_seen) > t                # item1 appears only after this row's time
early2 = test["item2"].map(first_seen) > t
print(f"item1 'not yet seen' at t: {early1.mean():.2%}, item2: {early2.mean():.2%}, "
      f"exactly one: {(early1 ^ early2).mean():.2%}")

item1 'not yet seen' at t: 12.97%, item2: 13.56%, exactly one: 25.47%


In [6]:
tr, pairs = make_pair_validation(train, include_timestamp=True, seed=0)

first_seen = tr.groupby("item")["timestamp"].min()
t = pairs["timestamp"]
early1 = pairs["item1"].map(first_seen) > t
early2 = pairs["item2"].map(first_seen) > t
one = early1 ^ early2                                   # exactly one item "not yet seen"
print(f"exactly one not-yet-seen: {one.mean():.2%}   (test: 25.47%)")

# within those rows: is the already-seen item the real one?
seen_item = np.where(early1, pairs["item2"], pairs["item1"])
print(f"rule accuracy on those rows: {(seen_item[one] == pairs['which'][one]).mean():.3f}")

exactly one not-yet-seen: 26.59%   (test: 25.47%)
rule accuracy on those rows: 0.970


In [7]:
tr, pairs = make_pair_validation(train, seed=0, include_timestamp=True)
calc_pair_accuracy(pairs, bpr_scorer(fit_bpr(tr, factors=5, regularization=0.01, learning_rate=0.01, iterations=100)))

pairs.head()

,user,item1,item2,which,timestamp
0,u8203075739,i1096367962,i3966153512,i1096367962,1500357411
1,u0999151525,i5186851769,i0169788189,i5186851769,1453107656
2,u6752727348,i7416016481,i1616614560,i7416016481,1448523563
3,u2237075147,i0004924020,i7351848776,i7351848776,1400055468
4,u6268264360,i6007667239,i3047831401,i3047831401,1412151117


In [8]:
fit = fit_bpr(tr, factors=5, regularization=0.01, learning_rate=0.01, iterations=100)
score = bpr_scorer(fit)
base = calc_pair_accuracy(pairs, bpr_scorer(fit))   

users = pairs["user"].to_numpy()
s1 = score(users, pairs["item1"].to_numpy())
s2 = score(users, pairs["item2"].to_numpy())
bpr_pred = np.where(s1 >= s2, pairs["item1"], pairs["item2"])       # BPR's choice per row

which = pairs["which"].to_numpy()
print(f"BPR overall:        {(bpr_pred == which).mean():.4f}")
print(f"BPR on rule rows:   {(bpr_pred[one] == which[one]).mean():.4f}")   # what the rule (0.970) has to beat
combined = np.where(one, seen_item, bpr_pred)
print(f"combined:           {(combined == which).mean():.4f}")

BPR overall:        0.8161
BPR on rule rows:   0.8647
combined:           0.8442


In [9]:
def run_grid_t4(grid: dict) -> pd.DataFrame:
    which = pairs["which"].to_numpy()
    users = pairs["user"].to_numpy()
    rows = []
    for values in tqdm(itertools.product(*grid.values()), total=prod(map(len, grid.values()))):
        params = dict(zip(grid, values))
        score = bpr_scorer(fit_bpr(tr, **params))
        s1 = score(users, pairs["item1"].to_numpy())
        s2 = score(users, pairs["item2"].to_numpy())
        bpr_pred = np.where(s1 >= s2, pairs["item1"], pairs["item2"])
        combined = np.where(one, seen_item, bpr_pred)
        rows.append({**params,
                     "bpr_acc": (bpr_pred == which).mean(),
                     "combined_acc": (combined == which).mean()})
    return pd.DataFrame(rows).sort_values("combined_acc", ascending=False)


In [10]:
grid = {
    "factors":        [5, 20, 50],
    "regularization": [0.001, 0.01, 0.1],
    "learning_rate":  [0.01],
    "iterations":     [100],
}

# results1 = run_grid_t4(grid)

# results1

In [11]:

grid = {
    "factors":        [50, 100, 200],
    "regularization": [0.003, 0.01, 0.03],
    "learning_rate":  [0.01],
    "iterations":     [100],
}

# results2 = run_grid_t4(grid)

# results2

In [12]:
TASK4_BPR_PARAMS = {"factors": 50, "regularization": 0.01, "learning_rate": 0.01, "iterations": 100}

bpr = bpr_scorer(fit_bpr(tr, **TASK4_BPR_PARAMS))
first_seen = clip_timestamps(tr["timestamp"]).groupby(tr["item"]).min()
item_tot = tr["item"].value_counts()

edges = time_bin_edges(tr["timestamp"], n_bins=50)
item_bin = (pd.Series(1, index=pd.MultiIndex.from_arrays([tr["item"], time_bins(tr["timestamp"], edges)]))
              .groupby(level=[0, 1]).size())

In [13]:
# def cand_features(users, items, ts):
#     t = clip_timestamps(ts).to_numpy(dtype=float)
#     fs = items.map(first_seen).to_numpy(dtype=float)              # NaN if unseen in tr
#     tot = items.map(item_tot).fillna(0).to_numpy(dtype=float)
#     rbin = time_bins(ts, edges)
#     in_bin = item_bin.reindex(pd.MultiIndex.from_arrays([items, rbin]), fill_value=0).to_numpy(dtype=float)
#     age = t - fs                                                   # seconds since first appearance
#     return pd.DataFrame({
#         "bpr":     bpr(users.to_numpy(), items.to_numpy()),
#         "not_yet": (age < 0).astype(float),
#         "log_age": np.log1p(np.clip(np.nan_to_num(age, nan=0.0), 0, None) / 86400),
#         "trend":   in_bin / (tot + 1),
#         "log_pop": np.log1p(tot),
#     })

F1 = cand_features(pairs["user"], pairs["item1"], pairs["timestamp"])
F2 = cand_features(pairs["user"], pairs["item2"], pairs["timestamp"])
X = (F1 - F2).to_numpy()
y = (pairs["which"] == pairs["item1"]).to_numpy().astype(int)

TypeError: cand_features() missing 1 required positional argument: 'ts'

In [ ]:
rng = np.random.default_rng(0)
half = rng.random(len(pairs)) < 0.5
which = pairs["which"].to_numpy()

lr = LogisticRegression(max_iter=1000).fit(X[half], y[half])
lr_pred = np.where(lr.predict(X) == 1, pairs["item1"], pairs["item2"])
print(f"LR on held-out half:        {(lr_pred[~half] == which[~half]).mean():.4f}")

users = pairs["user"].to_numpy()
s1, s2 = bpr(users, pairs["item1"].to_numpy()), bpr(users, pairs["item2"].to_numpy())
combined = np.where(one, seen_item, np.where(s1 >= s2, pairs["item1"], pairs["item2"]))
print(f"current rule on same half:  {(combined[~half] == which[~half]).mean():.4f}")

print(dict(zip(F1.columns, lr.coef_[0].round(3))))

LR on held-out half:        0.9001
current rule on same half:  0.8803
{'bpr': np.float64(1.221), 'not_yet': np.float64(-3.439), 'log_age': np.float64(-0.099), 'trend': np.float64(9.779), 'log_pop': np.float64(0.032)}


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(random_state=0).fit(X[half], y[half])
gb_pred = np.where(gb.predict(X) == 1, pairs["item1"], pairs["item2"])
print(f"GB on held-out half:  {(gb_pred[~half] == which[~half]).mean():.4f}   (LR: 0.9001, rule: 0.8803)")

GB on held-out half:  0.9026   (LR: 0.9001, rule: 0.8803)


In [ ]:
trend_tables = {}
for nb in [20, 50, 100]:
    e = time_bin_edges(tr["timestamp"], n_bins=nb)
    tbl = (pd.Series(1, index=pd.MultiIndex.from_arrays([tr["item"], time_bins(tr["timestamp"], e)]))
             .groupby(level=[0, 1]).size())
    trend_tables[nb] = (e, tbl)

def cand_features(users, items, ts):
    t = clip_timestamps(ts).to_numpy(dtype=float)
    fs = items.map(first_seen).to_numpy(dtype=float)
    tot = items.map(item_tot).fillna(0).to_numpy(dtype=float)
    age = t - fs
    feats = {
        "bpr":     bpr(users.to_numpy(), items.to_numpy()),
        "not_yet": (age < 0).astype(float),
        "log_age": np.log1p(np.clip(np.nan_to_num(age, nan=0.0), 0, None) / 86400),
        "log_pop": np.log1p(tot),
    }
    for nb, (e, tbl) in trend_tables.items():
        in_bin = tbl.reindex(pd.MultiIndex.from_arrays([items, time_bins(ts, e)]), fill_value=0).to_numpy(dtype=float)
        feats[f"trend_{nb}"] = in_bin / (tot + 1)
    return pd.DataFrame(feats)

In [ ]:
F1 = cand_features(pairs["user"], pairs["item1"], pairs["timestamp"])
F2 = cand_features(pairs["user"], pairs["item2"], pairs["timestamp"])
X = (F1 - F2).to_numpy()
y = (pairs["which"] == pairs["item1"]).to_numpy().astype(int)

In [ ]:
lr = LogisticRegression(max_iter=1000).fit(X[half], y[half])
lr_pred = np.where(lr.predict(X) == 1, pairs["item1"], pairs["item2"])

gb = HistGradientBoostingClassifier(random_state=0).fit(X[half], y[half])
gb_pred = np.where(gb.predict(X) == 1, pairs["item1"], pairs["item2"])

print(f"LR: {(lr_pred[~half] == which[~half]).mean():.4f}   (was 0.9001 with one trend)")
print(f"GB: {(gb_pred[~half] == which[~half]).mean():.4f}   (was 0.9026)")
print(dict(zip(F1.columns, lr.coef_[0].round(3))))

LR: 0.9018   (was 0.9001 with one trend)
GB: 0.9041   (was 0.9026)
{'bpr': np.float64(1.213), 'not_yet': np.float64(-2.926), 'log_age': np.float64(-0.03), 'log_pop': np.float64(0.031), 'trend_20': np.float64(3.489), 'trend_50': np.float64(3.795), 'trend_100': np.float64(2.531)}


In [ ]:
X_pkg = pair_features(fit_pair_context(tr, TASK4_BPR_PARAMS), pairs)
np.allclose(X_pkg, X), np.abs(X_pkg - X).max(axis=0)

(True, array([0., 0., 0., 0., 0., 0., 0.]))